###Build Driver Standings
#####Sources
    1. fact_session_results
    2. dim_drivers
#####Output Columns
    1. season
    2. driver_id
    3. driver_name
    4. nationality
    5. race starts
    6. total points
    7. number of wins
    8. number of podiums
    9. standing position

In [0]:
CREATE OR REPLACE VIEW formula1.gold.v_driver_standing
AS
WITH driven_session_summary
AS
    (SELECT r.season,
        d.driver_id,
        d.driver_name,
        d.nationality,
        COUNT(*) AS race_starts,
        SUM(r.points) AS total_points,
        COUNT_IF(r.is_win) AS number_of_wins,
        COUNT_IF(r.is_podium) AS number_of_podium
        FROM formula1.gold.fact_session_results r
        JOIN formula1.gold.dim_drivers d
        ON r.driver_id = d.driver_id
        WHERE r.season IS NOT NULL
        GROUP BY r.season,
                d.driver_id,
                d.driver_name,
                d.nationality
    )
SELECT season,
    driver_id,
    driver_name,
    nationality,
    RANK() OVER (PARTITION BY season ORDER BY total_points DESC, number_of_wins DESC) AS standing,
    race_starts,
    total_points,
    number_of_wins
FROM driven_session_summary